In [16]:
import sqlite3
import pandas as pd
import numpy as np

DB_PATH = "/Users/darraghdonnelly/dev/Database/recovered.db"
TEST_SIZE = 0.20

# connect to db
with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql_query(
        f"SELECT src, sex, age, distance_m, time_s FROM concat_results",
        conn,
    )

print(f"rows: {len(df)}")
df.head()

rows: 143442


,src,sex,age,distance_m,time_s
0,CherryBlossom2017,1,21,16093,3217.0
1,CherryBlossom2017,1,22,16093,3232.0
2,CherryBlossom2017,1,31,16093,3276.0
3,CherryBlossom2017,1,33,16093,3285.0
4,CherryBlossom2017,1,35,16093,3288.0


In [17]:
df["distance_bucket"] = (
    df["distance_m"].round().astype(int)
    .map({42195: "42k", 16093: "16k", 5000: "5k"})
)

df["distance_bucket"].head

<bound method NDFrame.head of 0         16k
1         16k
2         16k
3         16k
4         16k
         ... 
143437    16k
143438    16k
143439    16k
143440    16k
143441    16k
Name: distance_bucket, Length: 143442, dtype: str>

In [18]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# split dataset into train and test - stratified by distance
train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=42, stratify=df["distance_bucket"])

# store features in list
features = ["distance_m", "age", "sex"]

X_train = train_df[features]

# set target to time
y_train = train_df["time_s"]

# convert distances to labeled buckets (5k, 16k, 42k) so stratified k folding can be done
train_buckets = train_df["distance_bucket"]

X_test = test_df[features]
y_test = test_df["time_s"]

rf = RandomForestRegressor(n_estimators=200, max_depth=20, min_samples_leaf=2)

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# get mean abs error
cv_mae = -cross_val_score(
    rf,
    X_train,
    y_train,
    cv = skf.split(X_train, train_buckets),     # carry out strat kfold on training data for the 3 distances
    scoring="neg_mean_absolute_error"
)

# get r sqrd value
cv_r2 = cross_val_score(
    rf,
    X_train,
    y_train,
    cv=skf.split(X_train, train_buckets),
    scoring="r2"
)

print("CV MAE:", cv_mae.mean(), "with standard deviation of ±", cv_mae.std())
print("CV R2:", cv_r2.mean(), "with standard deviation of ±", cv_r2.std())


rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print("Test R2:", r2_score(y_test, y_pred))

CV MAE: 2191.4897659779967 with standard deviation of ± 4.452292332860376
CV R2: 0.730826033123385 with standard deviation of ± 0.0008878654700149397
Test R2: 0.7288083644753307


In [20]:
from sklearn.metrics import mean_absolute_error

# print out the mae mins for each bucket in test
test_eval = test_df.copy()
test_eval["pred_time_s"] = y_pred
test_eval["actual_time_s"] = y_test.values

for bucket in ["5k", "16k", "42k"]:
    bucket_rows = test_eval[test_eval["distance_bucket"] == bucket]
    mae_seconds = mean_absolute_error(bucket_rows["actual_time_s"], bucket_rows["pred_time_s"])
    mae_minutes = mae_seconds / 60

    print(bucket)
    print("MAE minutes:", mae_minutes)
    print()

5k
MAE minutes: 7.4334047167672574

16k
MAE minutes: 12.34464824664589

42k
MAE minutes: 45.162119065401285

